### using python kafka  kafka 

In [ ]:
from kafka import KafkaConsumer, TopicPartition, OffsetAndMetadata
from kafka.errors import KafkaError
import json
import logging

logging.basicConfig(level=logging.INFO)

# Create a Kafka consumer
consumer = KafkaConsumer(
    'example_topic',
    bootstrap_servers='localhost:9092',
    group_id='my-python-group',
    enable_auto_commit=False,  # Manual commit
    auto_offset_reset='earliest',  # Start from beginning if no offset
    value_deserializer=lambda m: m.decode('utf-8')
)

print("Consumer started...")

try:
    while True:
        # Poll for messages
        records = consumer.poll(timeout_ms=1000)

        for topic_partition, messages in records.items():
            for message in messages:
                print(f"Received: {message.value} | Offset: {message.offset}")

                # Process the message (put your logic here)
                # ...

            # ✅ Manual Synchronous Commit (reliable)
            try:
                consumer.commit()
                print(f"✔ Sync Commit done for partition {topic_partition}")
            except KafkaError as e:
                logging.error(f"❌ Sync Commit failed: {e}")

            # ✅ Manual Asynchronous Commit (faster, with callback)
            def on_commit_success(offsets, exception):
                if exception:
                    logging.error(f"❌ Async Commit failed: {exception}")
                else:
                    logging.info(f"✔ Async Commit successful: {offsets}")

            consumer.commit_async(callback=on_commit_success)

except KeyboardInterrupt:
    print("Stopping consumer...")

finally:
    try:
        # Do a final sync commit before exit for safety
        consumer.commit()
    except KafkaError as e:
        logging.error(f"Final commit failed: {e}")
    consumer.close()
    print("Consumer closed.")
